# APD benchmark — Operator workspace

You run the code; the chat model (Claude / ChatGPT / Gemini, free web chat) writes it. Data stays on Drive, mounted read-only. Your outputs are saved to `MyDrive/apd_outputs_<model>/` and survive Colab resets.

**Every session:** run cell 1. **New chat:** cell 2 (first time) or cell 4 (resume). **Then loop:** paste the model's cells into the empty cells at the bottom, run, copy the part between `COPY FROM HERE` and `END OF COPY` back into the chat.

**Before the first session**
- Drive: open the shared **`apd_benchmark`** folder → ⋮ → Organize → **Add shortcut** → My Drive.
- Colab: Tools → Settings → turn **off** AI-powered code completions / generation. Never use Colab's built-in Gemini panel. The code must come only from your chat.
- Your chat: turn off "use my conversations to improve the model" (or equivalent).

In [ ]:
#@title 1. Setup — run at the start of EVERY Colab session
# Mount Google Drive (NOT upload) and get the project code.
from google.colab import drive
drive.mount("/content/drive")

import os, sys, glob, shutil, subprocess
REPO = "/content/research"
if not os.path.exists(REPO):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")      # optional Colab secret (key icon on the left)
    except Exception:
        pass
    if token:
        subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/whiteraay/research.git", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/whiteraay/research.git"])
        print("code: cloned from GitHub")
    else:
        cands = ["/content/drive/MyDrive/apd_benchmark/code/research",
                 *glob.glob("/content/drive/.shortcut-targets-by-id/*/apd_benchmark/code/research")]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            raise SystemExit("No GITHUB_TOKEN secret and no code copy at MyDrive/apd_benchmark/code/research. "
                             "Add the apd_benchmark shortcut to My Drive (operators) or add a GITHUB_TOKEN secret.")
        shutil.copytree(src, REPO)
        print("code: copied from", src)
sys.path.insert(0, REPO + "/src")
from apd_bench import colab as apd

MODEL = "claude"     # <- operator 1: "claude" | operator 2: "gpt" | operator 3: "gemini"
apd.setup_workspace(MODEL)

## 2. First chat message: charter + execution protocol
Copy everything between the markers into a **new** chat. Wait for the model's acknowledgement.

In [ ]:
#@title 2. Print charter + protocol
apd.prompt(0)

## 3. Phase prompts
Send the next phase only after the model says **PHASE N COMPLETE** and `apd.ls()` shows its files.

In [ ]:
#@title 3. Print a phase prompt
PHASE = 1   # 1 audit · 2 proposal · 3 build · 4 evaluate · 5 affect · 6 report
apd.prompt(PHASE)

## 4. Usage limit hit / chat broke / new day
Open a **new** chat with the **same** model and paste this one message. It contains the charter, the protocol, STATE.md, decisions.md and the file list.

In [ ]:
#@title 4. Print resume packet
apd.resume_packet()

## 5. Log what happens in the chat
Record the exact model name the chat shows, and every limit, downgrade, refusal or new chat. These are part of the results.

In [ ]:
#@title 5. Log a chat event
EVENT = "start"        # start | limit | resume | downgrade | refusal | error | phase_done | gpu_on | gpu_unavailable
MODEL_SHOWN = ""       # exactly as the chat UI shows it, e.g. the model picker label
DETAIL = ""
apd.log_chat(EVENT, MODEL_SHOWN, DETAIL)

## 6. Paste the model's cells below
One model cell per Colab cell, in order, exactly as written. Add more cells with **+ Code**. Then paste the printed summary back into the chat.

In [ ]:
# CELL from the model

In [ ]:
# CELL from the model

In [ ]:
# CELL from the model

In [ ]:
# CELL from the model

## Before hand-off

In [ ]:
#@title Check outputs
apd.check_contract()
apd.ls()